**☀️ Welcome to ParkerNet Version 1.1 Notebook 2 of 3!**


With this notebook you will be able to do the following:


1.    Load in the individual .keras models for ParkerNet (each trained with a different seed and train split).
2.   Predict on a new dataset not used in training, validation, or testing parts of Notebook1. Additionally, SPANI datasets are also provided to test ParkerNet performance and cross-instrument capability. We upsampled the SPANI data from the "PSP_SWP_SPI_SF00_L3_MOM" files to match the cadence the network was trained on (0.8738 s) and synchronized with MAG data to create the MAG + SPANI datasets.
3.   Calculate the simple soft voting (simple averaging of predictions from all models) and weighted soft voting (weight models according to their AUC PRC (Area under Precision Recall curve) value).
4.    Create prediction datasets for data visualization and analysis.

***Note: We use a day in E5 for prediction here as a working example, but this notebook can be adjusted to load in the rest of the files used for prediction.***

The files you will need to run this notebook are:

PSP_E1toE7_July23_nonoise.txt (this is the dataset used for training and validation in Notebook1)

The following new datasets are used to predict on; each of these files contain approximately 1 day from each encounter near Perihelion.

MAG + SPC:

PSP_E1_ForPrediction.txt

PSP_E2_ForPrediction.txt

PSP_E4_ForPrediction.txt

PSP_E5_ForPrediction.txt

PSP_E6_ForPrediction.txt

PSP_E7_ForPrediction.txt

PSP_E8_ForPrediction.txt

MAG + SPANI: (E1 and E2 not used for SPANI - Solar wind mostly not in FOV of instrument)

PSP_E4_ForPrediction_SPANI.txt

PSP_E5_ForPrediction_SPANI.txt

PSP_E6_ForPrediction_SPANI.txt

PSP_E7_ForPrediction_SPANI.txt

PSP_E8_ForPrediction_SPANI.txt

You will also need the folliwing pre-trained ParkerNet models, which are provided for you:

| **Model Name** | **Model Name** | **Model Name** |
|--------------|--------------|--------------|
| ParkerNet_09172024_splitM_seed493.keras | ParkerNet_09172024_splitM_seed552.keras | ParkerNet_09172024_splitM_seed838.keras |
| ParkerNet_09172024_splitM_seed1022.keras | ParkerNet_08262024_splitM_seed123.keras | ParkerNet_08262024_splitM_seed324.keras |
| ParkerNet_08262024_splitM_seed369.keras | ParkerNet_08262024_splitM_seed564.keras | ParkerNet_08262024_splitM_seed641.keras |
| ParkerNet_08262024_splitM_seed910.keras | ParkerNet_08262024_splitM_seed1153.keras | ParkerNet_08262024_splitM_seed1187.keras |
| ParkerNet_08262024_splitM_seed775.keras | ParkerNet_08262024_splitM_seed1337.keras | ParkerNet_08262024_splitM_seed1886.keras |
| ParkerNet_08262024_splitM_seed1953.keras | ParkerNet_08262024_splitM_seed1962.keras | ParkerNet_09092024_splitN_seed493.keras |
| ParkerNet_09092024_splitN_seed552.keras | ParkerNet_09092024_splitN_seed838.keras | ParkerNet_09092024_splitN_seed1022.keras |
| ParkerNet_09092024_splitN_seed123.keras | ParkerNet_09092024_splitN_seed324.keras | ParkerNet_09092024_splitN_seed369.keras |
| ParkerNet_09092024_splitN_seed564.keras | ParkerNet_09092024_splitN_seed641.keras | ParkerNet_09092024_splitN_seed910.keras |
| ParkerNet_09092024_splitN_seed1153.keras | ParkerNet_09092024_splitN_seed1187.keras | ParkerNet_09092024_splitN_seed775.keras |
| ParkerNet_09092024_splitN_seed1337.keras | ParkerNet_09092024_splitN_seed1886.keras | ParkerNet_09092024_splitN_seed1953.keras |
| ParkerNet_09092024_splitN_seed1962.keras | ParkerNet_09032024_splitP_seed493.keras | ParkerNet_09032024_splitP_seed552.keras |
| ParkerNet_09032024_splitP_seed838.keras | ParkerNet_09032024_splitP_seed1022.keras | ParkerNet_09032024_splitP_seed123.keras |
| ParkerNet_09042024_splitP_seed324.keras | ParkerNet_09042024_splitP_seed369.keras | ParkerNet_09042024_splitP_seed564.keras |
| ParkerNet_09042024_splitP_seed641.keras | ParkerNet_09042024_splitP_seed910.keras | ParkerNet_09042024_splitP_seed1153.keras |
| ParkerNet_09042024_splitP_seed1187.keras | ParkerNet_09032024_splitP_seed775.keras | ParkerNet_09042024_splitP_seed1337.keras |
| ParkerNet_09042024_splitP_seed1886.keras | ParkerNet_09042024_splitP_seed1953.keras | ParkerNet_09032024_splitP_seed1962.keras |
| ParkerNet_10082024_splitN_seed1843.keras | ParkerNet_10082024_splitN_seed2816.keras | ParkerNet_10082024_splitN_seed983.keras |
| ParkerNet_10142024_splitN_seed2221.keras | ParkerNet_10142024_splitN_seed3060.keras | ParkerNet_10142024_splitN_seed3247.keras |
| ParkerNet_10142024_splitN_seed3364.keras | ParkerNet_10142024_splitN_seed3539.keras | ParkerNet_10142024_splitN_seed3871.keras |
| ParkerNet_10142024_splitN_seed400.keras | ParkerNet_10142024_splitN_seed4032.keras | ParkerNet_10142024_splitN_seed454.keras |
| ParkerNet_10162024_splitM_seed1843.keras | ParkerNet_10162024_splitM_seed2221.keras | ParkerNet_10162024_splitM_seed3060.keras |
| ParkerNet_10162024_splitM_seed3247.keras | ParkerNet_10162024_splitM_seed3364.keras | ParkerNet_10162024_splitM_seed3871.keras |
| ParkerNet_10162024_splitM_seed4032.keras | ParkerNet_10162024_splitM_seed983.keras | |

***Usage***: If you would like to predict on a dataset that is not in the traning data, nor in the files provided here. Make sure you follow the pre-processing steps in the publication and Notebook 1 exactly. The test set must have the same number of columns, in the same order, with the same time resolution. Once you have that, you can predict using each of the pre-trained models and compute a weighted average prediction. If no ground truth label can be found to compute AUC_PRC you may use another method to compute the weight. E.g.,  standard deviation accross each model for each sample, or Shannon entropy.




In [2]:
#load libraries
import pandas as pd
import os
import numpy as np
import tensorflow as tf
import keras
from keras import backend as K
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D,Activation, Dense, Dropout, Flatten, TimeDistributed, Bidirectional, LSTM, GlobalMaxPool1D, GlobalAveragePooling1D
from keras.optimizers import Adam
import matplotlib.pyplot as plt
from time import time
import seaborn as sns
import sklearn
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_curve
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, precision_score, recall_score, accuracy_score, auc
from keras import initializers
import tensorflow as tf
import random
from keras.models import load_model
import os
from sklearn.metrics import average_precision_score


Dependencies listed here

In [ ]:
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")
print(f"Seaborn: {sns.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")

Pandas: 2.2.2
NumPy: 2.0.2
TensorFlow: 2.18.0
Keras: 3.8.0
Seaborn: 0.13.2
Scikit-learn: 1.6.1


If files need to be uploaded from your local machine, provide the path.

In [3]:
from google.colab import drive
drive.mount('/content/MyDrive')

Mounted at /content/MyDrive


In [4]:
cd MyDrive/MyDrive

/content/MyDrive/MyDrive


In [5]:

# Load the dataset used for training
df = pd.read_csv('PSP_E1toE7_July23_nonoise.txt', sep="\t", parse_dates=['Datetime'], index_col='Datetime')


df['Class'] = df['Class'].map({True: 1, False: 0}).astype(int)

# Drop unnecessary columns
df.pop("indices")
df.pop("Encounter")
X_HCI_train = df.pop("X_HCI")
Y_HCI_train = df.pop("Y_HCI")
Z_HCI_train = df.pop("Z_HCI")
df.pop("Dist")

# Preview the dataset
df.head()




,B_r,Bmag,B_t,B_n,V_r,V_t,V_n,Vmag,V_nr,ProtonDensity,Class
Datetime,,,,,,,,,,,
2018-11-04 00:00:00.000,-66.058226,70.255204,-4.140612,-23.557585,267.886,-28.3115,-39.9285,272.321015,48.947177,391.665,0
2018-11-04 00:00:00.873,-65.699724,70.171543,-3.381591,-24.416317,263.905,-69.5283,-43.8579,276.411919,82.205230,536.104,0
2018-11-04 00:00:01.747,-66.442084,70.452417,-8.469396,-21.846322,266.871,-41.5544,-24.7354,271.217143,48.359158,367.138,0
2018-11-04 00:00:02.621,-65.771892,70.127234,-15.798794,-18.500954,260.689,-57.2835,-41.0414,270.045460,70.468403,504.809,0
2018-11-04 00:00:03.495,-64.232595,70.390542,-18.842718,-21.770487,267.265,-26.0570,-45.9163,272.429540,52.794639,394.882,0


Some information about the prediction files:

-The files meant for prediction has a class label made from using Huang et al. (2023) catalog data. This class label is made simply by making a binary mask that indicates a 1 where Huang et al. (2023) has a swithback (LQP start to TQP end). This class column is used to calculate the AUC_PRC later for the weighted voting section.

-The file for each encounter spans about one day in time.

-When loading in the files using the code below, make sure to change the number of the encounter: for example if loading in 'PSP_E1_ForPrediction.txt' make sure to change df_EX to df_E1 etc.

# ====== Loading in data for prediction ======

We will load 1) A MAG + SPC dataset and 2) A MAG + SPANI dataset

In [6]:
# Read in the dataset for predicting (provided as PSP_EX_yyyymmdd.txt) using the MAG + SPC data
df_E5 = pd.read_csv('PSP_E5_ForPrediction.txt', sep="\t", parse_dates=['Datetime'], index_col='Datetime')#change df_E7 to df_E1, df_E2 etc when you load the corresponding file for each encounter (e.g., PSP_E1_forPrediction.txt)

# Separate the 'Class_Huang_Ex' column into df_gt
df_gt_E5 = df_E5[['Class_Huang_E5_Range']].copy()

pd.set_option('future.no_silent_downcasting', True)
df_gt_E5['Class_Huang_E5_Range'] = df_gt_E5['Class_Huang_E5_Range'].replace({True: 1, False: 0}).infer_objects(copy=False).astype(int)
# Create df_test by dropping the 'Class_Huang_Ex' column and other columns not needed. df_test has only features to predict on. The Class_Huang column has been removed and saved as a ground_truth check for later use in calculating AUC_PRC
df_test_E5= df_E5.drop(columns=['Class_Huang_E5_Range'])

#UNCOMMENT FOR NON SPANI DATASET!!!!!
df_test_E5.pop("X_HCI")
df_test_E5.pop("Y_HCI")
df_test_E5.pop("Z_HCI")

,Z_HCI
Datetime,
2020-06-02 00:00:00.000000000,606485
2020-06-02 00:00:00.873799999,606480
2020-06-02 00:00:01.747599999,606476
2020-06-02 00:00:02.621399999,606472
2020-06-02 00:00:03.495199999,606468
...,...
2020-06-02 23:59:56.974996393,197255
2020-06-02 23:59:57.848796393,197251
2020-06-02 23:59:58.722596393,197247


In [ ]:
# Read in the dataset for predicting (provided as PSP_EX_yyyymmdd.txt) using the MAG + SPANI data
df_E5_SPANI = pd.read_csv('PSP_E5_ForPrediction_SPANI.txt', sep="\t", parse_dates=['Datetime'], index_col='Datetime')#change df_EX to df_E1, df_E2 etc when you load the corresponding file for each encounter (e.g., PSP_E1_forPrediction.txt)

# Separate the 'Class_Huang_Ex' column into df_gt
df_gt_E5_SPANI = df_E5_SPANI[['Class_Huang_E5_Range']].copy()

pd.set_option('future.no_silent_downcasting', True)
df_gt_E5_SPANI['Class_Huang_E5_Range'] = df_gt_E5_SPANI['Class_Huang_E5_Range'].replace({True: 1, False: 0}).infer_objects(copy=False).astype(int)
# Create df_test by dropping the 'Class_Huang_Ex' column and other columns not needed. df_test has only features to predict on. The Class_Huang column has been removed and saved as a ground_truth check for later use in calculating AUC_PRC
df_test_E5_SPANI= df_E5_SPANI.drop(columns=['Class_Huang_E5_Range'])



df_gt_EX is the ground truth, in each prediction file there will be a column called Class_Huang_EX_Range. We have used the Huang et al. (2023) catalog to create the ground truth flags using LQP start to TQP end.  This column is used to calculate the AUC_PRC used later in the notebook in the ensemble averaging section.

We will use split M's stats to do the z-scaling for the prediction set

In [7]:
#test M`, train: E1 to E4, val:E5-E6 EOD Sept 23, test 1:E6 Sept 24 + E7 , test2: E15
Xtrain = df.iloc[0:1615023,:-1] #everything but the last column (split1)
ytrain = df.iloc[0:1615023,-1]#only pick the last column
Xval = df.iloc[1615023:1788065,:-1] #everything but the last column (split 1)
yval = df.iloc[1615023:1788065,-1]#only pick the last column
Xtest1 = df.iloc[1788065:,:-1] #everything but the last column
ytest1 = df.iloc[1788065:,-1]#only pick the last column

z-scaling

In [8]:
Xpred = df_test_E5 #pred dataset has only input variables so use it as it is, replace with df_test_EX or df_test_EX_SPANI after loading as described above
train_mean = Xtrain.mean()
train_std = Xtrain.std()

train_df = (Xtrain - train_mean) / train_std
val_df = (Xval - train_mean) / train_std
test_df = (Xtest1 - train_mean) / train_std
pred_df = (Xpred - train_mean) / train_std

Creating sequences

In [9]:
def split_sequences_nolags(dataset,labels, time_steps):
    data_X, data_Y = [], []
    for i in range(len(dataset)-time_steps):
        a = dataset.iloc[i:(i+time_steps)]
        data_X.append(a)
        data_Y.append(labels.iloc[i:i + time_steps])
    return np.array(data_X), np.array(data_Y)

Since the prediction sets contain no labels you will need to create sequences of the features only and not the labels

In [10]:
#use this for test set with no labels, so since the prediction set has no labels yet you will use this sequence function which is simply a modified version of the split_sequences_nolags function above
def split_sequences_nolags_test(dataset, time_steps):
    data_X = []
    for i in range(len(dataset)-time_steps):
        a = dataset.iloc[i:(i+time_steps)]
        data_X.append(a)
        #data_Y.append(labels.iloc[i:i + time_steps])
    return np.array(data_X)

In [11]:
pred_features_new = split_sequences_nolags_test(pred_df,50) #note that the no class label sequence function is used here. The class label is removed and used later for calculating AUC_PRC


# ====== Loading in Pre-Trained ParkerNet Models ======

You need to use this part of the code for the models to get loaded correctly. The "@" function here is a decorator function which tells Keras how to use the custom binary loss function when loading the pre-trained models. It is used to register a custom function so Keras can reconstruct the function. Without using this, you will not be able to load the models.

In [12]:
POS_WEIGHT = 200
POS_WEIGHT = POS_WEIGHT
@keras.saving.register_keras_serializable()
def weighted_binary_crossentropy(target, output):
    """
    Weighted binary crossentropy between an output tensor
    and a target tensor. POS_WEIGHT is used as a multiplier
    for the positive targets.

    Combination of the following functions:
    * keras.losses.binary_crossentropy
    * keras.backend.tensorflow_backend.binary_crossentropy
    * tf.nn.weighted_cross_entropy_with_logits
    """
    # transform back to logits
    _epsilon = tf.convert_to_tensor(tf.keras.backend.epsilon(), output.dtype.base_dtype)
    output = tf.clip_by_value(output, _epsilon, 1 - _epsilon)
    output = tf.math.log(output / (1 - output))
    loss = tf.nn.weighted_cross_entropy_with_logits(labels=target, logits=output, pos_weight=POS_WEIGHT)

    return tf.reduce_mean(loss, axis=-1)

Loading Pre-trained ParkerNet Models. Here we will load each of the pre-trained models (with differing train splits, and seeds) and put each of their predictions in the dataset into a dataframe. This will allow us to keep a record of how well each model predicts. It will also allow us to calculate an average or "ensemble" prediction.

Note to user: it will take about 20 minutes to predict on a one day dataset using all 71 models.

In [13]:
#USE THIS
# Function to extract the model name from the filename without the '.keras' extension
def get_model_name(filepath):
    filename = os.path.basename(filepath)
    filename = os.path.splitext(filename)[0]
    return "_".join(filename.split('_')[2:])

# Load the pre-trained models and their filenames
model_paths = [
    'ParkerNet_09172024_splitM_seed493.keras',
    'ParkerNet_09172024_splitM_seed552.keras',
    'ParkerNet_09172024_splitM_seed838.keras',
    'ParkerNet_09172024_splitM_seed1022.keras',
    'ParkerNet_08262024_splitM_seed123.keras',
    'ParkerNet_08262024_splitM_seed324.keras',
    'ParkerNet_08262024_splitM_seed369.keras',
    'ParkerNet_08262024_splitM_seed564.keras',
    'ParkerNet_08262024_splitM_seed641.keras',
    'ParkerNet_08262024_splitM_seed910.keras',
    'ParkerNet_08262024_splitM_seed1153.keras',
    'ParkerNet_08262024_splitM_seed1187.keras',
    'ParkerNet_08262024_splitM_seed775.keras',
    'ParkerNet_08262024_splitM_seed1337.keras',
    'ParkerNet_08262024_splitM_seed1886.keras',
    'ParkerNet_08262024_splitM_seed1953.keras',
    'ParkerNet_08262024_splitM_seed1962.keras',
    'ParkerNet_09092024_splitN_seed493.keras',
    'ParkerNet_09092024_splitN_seed552.keras',
    'ParkerNet_09092024_splitN_seed838.keras',
    'ParkerNet_09092024_splitN_seed1022.keras',
    'ParkerNet_09092024_splitN_seed123.keras',
    'ParkerNet_09092024_splitN_seed324.keras',
    'ParkerNet_09092024_splitN_seed369.keras',
    'ParkerNet_09092024_splitN_seed564.keras',
    'ParkerNet_09092024_splitN_seed641.keras',
    'ParkerNet_09092024_splitN_seed910.keras',
    'ParkerNet_09092024_splitN_seed1153.keras',
    'ParkerNet_09092024_splitN_seed1187.keras',
    'ParkerNet_09092024_splitN_seed775.keras',
    'ParkerNet_09092024_splitN_seed1337.keras',
    'ParkerNet_09092024_splitN_seed1886.keras',
    'ParkerNet_09092024_splitN_seed1953.keras',
    'ParkerNet_09092024_splitN_seed1962.keras',
    'ParkerNet_09032024_splitP_seed493.keras',
    'ParkerNet_09032024_splitP_seed552.keras',
    'ParkerNet_09032024_splitP_seed838.keras',
    'ParkerNet_09032024_splitP_seed1022.keras',
    'ParkerNet_09032024_splitP_seed123.keras',
    'ParkerNet_09042024_splitP_seed324.keras',
    'ParkerNet_09042024_splitP_seed369.keras',
    'ParkerNet_09042024_splitP_seed564.keras',
    'ParkerNet_09042024_splitP_seed641.keras',
    'ParkerNet_09042024_splitP_seed910.keras',
    'ParkerNet_09042024_splitP_seed1153.keras',
    'ParkerNet_09042024_splitP_seed1187.keras',
    'ParkerNet_09032024_splitP_seed775.keras',
    'ParkerNet_09042024_splitP_seed1337.keras',
    'ParkerNet_09042024_splitP_seed1886.keras',
    'ParkerNet_09042024_splitP_seed1953.keras',
    'ParkerNet_09032024_splitP_seed1962.keras',
    'ParkerNet_10082024_splitN_seed1843.keras',
    'ParkerNet_10082024_splitN_seed2816.keras',
    'ParkerNet_10082024_splitN_seed983.keras',
    'ParkerNet_10142024_splitN_seed2221.keras',
    'ParkerNet_10142024_splitN_seed3060.keras',
    'ParkerNet_10142024_splitN_seed3247.keras',
    'ParkerNet_10142024_splitN_seed3364.keras',
    'ParkerNet_10142024_splitN_seed3539.keras',
    'ParkerNet_10142024_splitN_seed3871.keras',
    'ParkerNet_10142024_splitN_seed400.keras',
    'ParkerNet_10142024_splitN_seed4032.keras',
    'ParkerNet_10142024_splitN_seed454.keras',
    'ParkerNet_10162024_splitM_seed1843.keras',
    'ParkerNet_10162024_splitM_seed2221.keras',
    'ParkerNet_10162024_splitM_seed3060.keras',
    'ParkerNet_10162024_splitM_seed3247.keras',
    'ParkerNet_10162024_splitM_seed3364.keras',
    'ParkerNet_10162024_splitM_seed3871.keras',
    'ParkerNet_10162024_splitM_seed4032.keras',
    'ParkerNet_10162024_splitM_seed983.keras'
]

# Create an empty DataFrame to store the predictions
df_allprobs = pd.DataFrame()

# Function to predict and add the results to the dataframe
def predict_and_store(model, model_name, df, pred_features_new):
    probs_pred = model.predict(pred_features_new, batch_size=1024)
    final_classification = np.mean(probs_pred, axis=1)
    df[model_name] = pd.Series(np.squeeze(final_classification))
    return df

# ✅ Load models with custom loss, predict, and store
for model_path in model_paths:
    model = load_model(model_path, custom_objects={
        'weighted_binary_crossentropy': weighted_binary_crossentropy
    })

    model_name = get_model_name(model_path)
    pred_column_name = f'Preds_{model_name}'
    df_allprobs = predict_and_store(model, pred_column_name, df_allprobs, pred_features_new)

# Final processing
df_gt_E5.reset_index(drop=True, inplace=True)
df_allprobs.reset_index(drop=True, inplace=True)

df_gt_allpreds_E5 = pd.concat([df_gt_E5, df_allprobs], axis=1, join='inner')
display(df_gt_allpreds_E5)


97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 157ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 17s 163ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 162ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 161ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 161ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 160ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 159ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 161ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 17s 168ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 162ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 162ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 163ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 163ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 162ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 162ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 161ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 164ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 160ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 161ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 159ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 162ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 162ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 16s 161ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 17s 167

,Class_Huang_E5_Range,Preds_splitM_seed493,Preds_splitM_seed552,Preds_splitM_seed838,Preds_splitM_seed1022,Preds_splitM_seed123,Preds_splitM_seed324,Preds_splitM_seed369,Preds_splitM_seed564,Preds_splitM_seed641,...,Preds_splitN_seed4032,Preds_splitN_seed454,Preds_splitM_seed1843,Preds_splitM_seed2221,Preds_splitM_seed3060,Preds_splitM_seed3247,Preds_splitM_seed3364,Preds_splitM_seed3871,Preds_splitM_seed4032,Preds_splitM_seed983
0,0,0.570572,0.232200,0.485185,0.218807,0.181046,0.279613,0.283489,0.359092,0.256064,...,0.143090,0.249585,0.421048,0.204260,0.213318,0.252517,0.476842,0.347327,0.288870,0.194782
1,0,0.581111,0.231295,0.486473,0.220719,0.181822,0.283966,0.283273,0.363969,0.254928,...,0.146835,0.249481,0.420282,0.204879,0.216148,0.257787,0.488174,0.347868,0.295282,0.193620
2,0,0.585266,0.230741,0.490033,0.220378,0.181224,0.287258,0.285186,0.361671,0.250972,...,0.148179,0.253700,0.421147,0.207727,0.214395,0.258325,0.492783,0.346804,0.303191,0.192192
3,0,0.582024,0.230883,0.487392,0.220574,0.181777,0.289551,0.286086,0.356933,0.250042,...,0.147400,0.254817,0.419676,0.206500,0.218894,0.261805,0.488114,0.345444,0.304157,0.190043
4,0,0.592703,0.233855,0.484906,0.222088,0.181291,0.296363,0.284925,0.357670,0.247380,...,0.149592,0.254164,0.416427,0.205087,0.222487,0.265805,0.494479,0.347764,0.313193,0.192227
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98825,0,0.273151,0.339644,0.424701,0.240795,0.344574,0.194448,0.341039,0.235337,0.204121,...,0.115335,0.235412,0.530894,0.209684,0.164287,0.191585,0.242145,0.315794,0.205994,0.248127
98826,0,0.273306,0.337206,0.423767,0.241523,0.344157,0.193528,0.340864,0.235472,0.204277,...,0.115170,0.235799,0.529546,0.211273,0.164102,0.192284,0.243171,0.315514,0.205650,0.247905
98827,0,0.273500,0.339445,0.421918,0.241498,0.343255,0.193556,0.340417,0.235463,0.204568,...,0.115087,0.236316,0.527849,0.210497,0.164795,0.192659,0.243371,0.316356,0.205387,0.247572
98828,0,0.273587,0.342927,0.421311,0.242238,0.342245,0.194405,0.340432,0.234716,0.204436,...,0.114806,0.235751,0.526116,0.210338,0.165502,0.192906,0.244969,0.318514,0.204683,0.247129


Use the code below to write all 71 model predictions along with the ground truth to a csv file for later use.

In [ ]:
#df_gt_allpreds_EX.to_csv('ParkerNet_allModel_predictions_EX_yyyymmdd.csv', index=False) #use this to save your predictions for each model into a csv file

# ====== Ensemble Prediction ======

### Simple Soft Voting (Average Prediction)

We will now calculate the average prediction from all the predictions (Simple Soft Voting) ; this is not used in the analysis but is calculated in case anyone wanted to see the difference between simply averaging models predictions and using a weighted average.

In [ ]:

# Dynamically extract all model prediction columns (everything except the ground truth)
model_columns = [col for col in df_gt_allpreds_E5.columns if col != 'Class_Huang_E5_Range']

# Ground truth labels
y_true = df_gt_allpreds_E5['Class_Huang_E5_Range'].values

# Step 1: Simple soft voting (equal weight for all models)
# Initialize the final predictions column
df_gt_allpreds_E5['SimpleSoftVoting_Predictions'] = np.zeros_like(y_true, dtype=float)

# Simply sum the probabilities of all models and divide by the number of models (average)
num_models = len(model_columns)
for model in model_columns:
    df_gt_allpreds_E5['SimpleSoftVoting_Predictions'] += df_gt_allpreds_E5[model]

# Normalize by the number of models to get the average (equal weight soft voting)
df_gt_allpreds_E5['SimpleSoftVoting_Predictions'] /= num_models

# The 'SimpleSoftVoting_Predictions' column now contains the soft voting ensemble probabilities
print(df_gt_allpreds_E5[['Class_Huang_E5_Range', 'SimpleSoftVoting_Predictions']].head())

   Class_Huang_E5_Range  SimpleSoftVoting_Predictions
0                     0                      0.466471
1                     0                      0.466043
2                     0                      0.465583
3                     0                      0.465102
4                     0                      0.464734


### Weighted Average Prediction using AUC_PRC as weight


Now we will calculate the weighted soft voting using the Huang class label to calculate the AUC_PRC (Area under the precision-recall curve). This is calculated and saved as "Weighted_Voting_AUC_PRC" in the dataset. This is the column that has been used in the analysis for the paper. If you scroll to the right on the displayed dataframe, you will now see this column added to the end.

In [ ]:

#from sklearn.metrics import average_precision_score

# Define the target column and identify model prediction columns
target_column = 'Class_Huang_E5_Range'
model_columns = [col for col in df_gt_allpreds_E5.columns if col not in [target_column, 'SimpleSoftVoting_Predictions']]

# Calculate AUC-PRC for each model
auc_prc_scores = {}
for model in model_columns:
    auc_prc = average_precision_score(df_gt_allpreds_E5[target_column], df_gt_allpreds_E5[model])
    auc_prc_scores[model] = auc_prc

# Normalize AUC-PRC scores for weighting
total_auc = sum(auc_prc_scores.values())
weights = {model: auc / total_auc for model, auc in auc_prc_scores.items()}

# Calculate weighted soft voting predictions
df_gt_allpreds_E5['Weighted_Voting_AUC_PRC'] = sum(
    df_gt_allpreds_E5[model] * weight for model, weight in weights.items()
)


display(df_gt_allpreds_E5)

,Class_Huang_E5_Range,Preds_splitM_seed493,Preds_splitM_seed552,Preds_splitM_seed838,Preds_splitM_seed1022,Preds_splitM_seed123,Preds_splitM_seed324,Preds_splitM_seed369,Preds_splitM_seed564,Preds_splitM_seed641,...,Preds_splitM_seed1843,Preds_splitM_seed2221,Preds_splitM_seed3060,Preds_splitM_seed3247,Preds_splitM_seed3364,Preds_splitM_seed3871,Preds_splitM_seed4032,Preds_splitM_seed983,SimpleSoftVoting_Predictions,Weighted_Voting_AUC_PRC
0,0,0.562663,0.163775,0.472671,0.186366,0.183859,0.205806,0.279678,0.382518,0.184835,...,0.423834,0.172812,0.172734,0.204234,0.460900,0.239714,0.228356,0.197336,0.466471,0.235819
1,0,0.562870,0.163165,0.472768,0.186400,0.183642,0.205221,0.279609,0.382173,0.184208,...,0.423692,0.172678,0.172139,0.204161,0.459217,0.238907,0.228177,0.197366,0.466043,0.235591
2,0,0.564378,0.162509,0.472748,0.186412,0.183749,0.204853,0.279696,0.380843,0.183546,...,0.423542,0.172622,0.172109,0.203964,0.457277,0.237602,0.227451,0.197480,0.465583,0.235349
3,0,0.563928,0.162062,0.472888,0.186473,0.183720,0.204132,0.279513,0.379764,0.182958,...,0.423332,0.172458,0.171690,0.203713,0.455576,0.236587,0.227131,0.197393,0.465102,0.235095
4,0,0.564348,0.161447,0.472766,0.186551,0.183877,0.203475,0.279519,0.379170,0.182326,...,0.423107,0.172348,0.171446,0.203396,0.454571,0.235326,0.226849,0.197379,0.464734,0.234904
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98825,0,0.253389,0.158275,0.427633,0.187299,0.164280,0.178509,0.279608,0.247296,0.179634,...,0.264197,0.171837,0.170392,0.170062,0.203294,0.189342,0.163353,0.171028,0.395237,0.199088
98826,0,0.253245,0.158071,0.427610,0.187322,0.164304,0.178448,0.279536,0.247267,0.179715,...,0.264091,0.171808,0.170435,0.170025,0.203621,0.189304,0.163117,0.171114,0.395179,0.199061
98827,0,0.253019,0.157988,0.427599,0.187331,0.164333,0.178412,0.279501,0.247247,0.179793,...,0.263995,0.171798,0.170613,0.169976,0.203985,0.189192,0.162877,0.171333,0.395156,0.199052
98828,0,0.252647,0.158056,0.427534,0.187306,0.164386,0.178440,0.279373,0.247045,0.179953,...,0.263754,0.171803,0.170744,0.169906,0.204471,0.189172,0.162800,0.171460,0.395190,0.199073


Save the dataframe now containing the ensemble prediciton for analysis (will need this for Notebook 3 of 3) using the code below.

In [ ]:
#df_gt_allpreds_EX.to_csv('ParkerNet_allModel_averaged_predictions_EX.csv', index=False)

***Now you know how to load in pre-trained keras models, predict on a dataset using all the models, and make an ensemble prediction.***